# Review an AI-written ROAS analysis

*Data Analysis in Python · Lesson 14 final exercise*

A colleague asked an AI assistant for Q1 2025's ROAS by platform and conversion rate by channel group. The notebook below is what it wrote, and it runs without a single error. Its summary recommends moving budget to Meta. Before anyone acts on it, review it: three lines are wrong.

**Before you start**, choose **File → Save a copy in Drive**, and work in the copy that opens. This workbook is shared with every student, so Colab doesn't save your changes to it.

You'll fix the AI's cells where they are: change the lines that are wrong, and leave the rest. Your own cells go in the **Your checks** section, above the **Check your work** cell. Each step says which topic showed how, and if you're stuck, open its hint under the **Check your work** cell.

1. Choose **Runtime → Run all**, and read the AI's notebook from the top, with its summary.
2. **Let your checks find the problems.** In the first cell of **Your checks**, paste topic 14.3.'s `validate_report(sessions, report, start, end)` from your 14.3. workbook, and run it on the AI's tables: `validate_report(sessions, channels, START, END)`. Two of its four checks fail. Each one points to a problem.
3. **Find the missing days.** `every day is there` failed, so some days of Q1 2025 aren't in the `sessions` table. Count the days in each month (the hint has the line), and find the month that's short. Then look at that month's export as it comes, before the AI's cell reads it: how are its dates written? Compare that with how the AI's first cell reads dates, and fix that cell, so it reads that month with the right format (topic 3.2.).
4. **Fix the rates.** `rates from totals` failed. Look at how the AI's channels cell calculates `conversion_rate`, and change it so each channel group's rate is its transactions divided by its sessions (topic 4.4.).
5. **Find the duplicated revenue.** `validate_report()` doesn't look at ROAS, so ask topic 14.2.'s first question yourself: did rows appear? Count the `spend` table's rows for each platform, and the number of different date and campaign pairs for each platform. One platform has more rows than pairs: its export has a row per ad set. Fix the AI's platforms cell, so the spend is totalled by date, platform and campaign before the merge, and the merge stops if a date and campaign appear more than once (topic 5.4.).
6. **Make the checks stop the notebook.** Below `validate_report()`, create `checks` for the `sessions` and `channels` tables, and `assert` that they all pass.
7. **Rewrite the summary** with the right numbers.
8. Choose **Runtime → Restart session and run all**, and read the **Check your work** cell's output.

**Expected output**: The `sessions` table has 5,297 rows, every day of Q1 2025, with 324,563 sessions and \$655,182.61 of revenue. The `channels` table has 9 channel groups, with Email converting best at 6.22%. The `platforms` table's ROAS is Google Ads 2.47, Meta 2.33 and YouTube 1.89. Every check in `checks` passes.

## Parameters

In [ ]:
import pandas as pd

DATA_URL = "https://storage.googleapis.com/simmer-python-course-data/simmer-store/v1/"
MONTHS = ["2025-01", "2025-02", "2025-03"]
START, END = "2025-01-01", "2025-03-31"  # Q1 2025

In [ ]:
# The team's channel groups, from lesson 4. Run it, and leave it as it is: the AI's first cell uses it.
SOCIAL_SITES = {"facebook.com", "m.facebook.com", "l.facebook.com", "lm.facebook.com", "instagram.com", "l.instagram.com"}


def clean(value):
    """Return the value lowercased and stripped, or "" for None."""
    if value is None:
        return ""
    return value.strip().lower()


def channel_group(source, medium):
    """Return the channel group of a session's source and medium, with Foundations topic 12.3's rules."""
    source = clean(source)
    medium = clean(medium)
    if source == "(direct)" and medium == "(none)":
        return "Direct"
    if medium == "email":
        return "Email"
    if medium == "cpc":
        return "Paid Search"
    if medium == "display":
        return "Display"
    if medium == "paid_social":
        return "Paid Social"
    if medium == "video":
        return "Paid Video"
    if medium == "organic":
        return "Organic Search"
    if medium == "referral" and source in SOCIAL_SITES:
        return "Organic Social"
    if medium == "referral":
        return "Referral"
    return "Unassigned"


def row_channel_group(row):
    """Return the channel group of one row of the sessions table."""
    return channel_group(row["session_source"], row["session_medium"])

## The AI's notebook

Your colleague's prompt to the assistant was: *"Here are our GA4 session exports and our ad platforms' spend files. Write a notebook that reports Q1 2025's conversion rate by channel group and ROAS by platform, and summarize where we should move budget."* Every cell from here to the summary is what the assistant wrote.

In [ ]:
# Load the Q1 2025 session exports and clean them
frames = []
for month in MONTHS:
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    # Parse dates flexibly, so every export's date format works
    export["date"] = pd.to_datetime(export["date"], format="mixed")
    # Some exports store revenue as currency text, such as "$1,234.50"
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    frames.append(export)

raw = pd.concat(frames, ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()  # The monthly exports overlap by two days
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)  # The team's channel groups
sessions.shape

In [ ]:
# Conversion rate by channel group: average the daily rates, so one unusual day can't skew the result
daily = sessions.groupby(["channel_group", "date"], as_index=False)[["sessions", "transactions", "revenue"]].sum()
daily["conversion_rate"] = daily["transactions"] / daily["sessions"]
channels = daily.groupby("channel_group", as_index=False).agg(
    sessions=("sessions", "sum"),
    transactions=("transactions", "sum"),
    revenue=("revenue", "sum"),
    conversion_rate=("conversion_rate", "mean"),
)
channels = channels.sort_values("revenue", ascending=False).reset_index(drop=True)
channels

In [ ]:
# One spend table from the three platforms' exports
google = pd.read_csv(DATA_URL + "google_ads_spend.csv", parse_dates=["date"])
meta = pd.read_csv(DATA_URL + "meta_ads_spend.csv", parse_dates=["Reporting starts"])
youtube = pd.read_csv(DATA_URL + "youtube_spend.csv", parse_dates=["Day"])

google = google.rename(columns={"campaign_name": "campaign"})
google["spend"] = google["cost_micros"] / 1_000_000  # Google Ads reports cost in millionths of a dollar
google["platform"] = "Google Ads"
meta = meta.rename(columns={"Reporting starts": "date", "Campaign name": "campaign", "Amount spent (USD)": "spend"})
meta["platform"] = "Meta"
youtube = youtube.rename(columns={"Day": "date", "Campaign": "campaign", "Cost": "spend"})
youtube["platform"] = "YouTube"

columns = ["date", "platform", "campaign", "spend"]
spend = pd.concat([google[columns], meta[columns], youtube[columns]], ignore_index=True)
spend = spend[spend["date"].between(START, END)].reset_index(drop=True)
spend["campaign"] = spend["campaign"].str.strip().str.lower()  # Match the campaign names in GA4
spend.groupby("platform")["spend"].sum()

In [ ]:
# Revenue per date and campaign from GA4, joined to the spend for ROAS
revenue = sessions.groupby(["date", "session_campaign"], as_index=False)[["sessions", "transactions", "revenue"]].sum()
revenue = revenue.rename(columns={"session_campaign": "campaign"})
revenue["campaign"] = revenue["campaign"].str.strip().str.lower()

campaign_days = spend.merge(revenue, on=["date", "campaign"], how="left")
platforms = campaign_days.groupby("platform")[["spend", "revenue"]].sum()
platforms["roas"] = platforms["revenue"] / platforms["spend"]
platforms.round(2)

**Summary.** In Q1 2025, Meta returned \$5.83 for every dollar it spent, more than twice Google Ads' \$2.37 and three times YouTube's \$1.66. Meta is clearly the most efficient platform, and I'd move budget to it from YouTube first, then from Google Ads. Among channel groups, Email converts best, at 6.54% of sessions.

## Your checks

Your cells go here: `validate_report()` first (step 2), then the cells you use to find the problems, and the `assert` (step 6).

In [ ]:
# The AI's first cell, fixed: March 2025's export writes its dates day first
DAY_FIRST_MONTHS = ["2025-03"]
frames = []
for month in MONTHS:
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    # Some exports store revenue as currency text, such as "$1,234.50"
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    frames.append(export)

raw = pd.concat(frames, ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()  # The monthly exports overlap by two days
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)  # The team's channel groups
sessions.shape

In [ ]:
# The AI's channels cell, fixed: rates from totals
channels = sessions.groupby("channel_group", as_index=False)[["sessions", "transactions", "revenue"]].sum()
channels["conversion_rate"] = channels["transactions"] / channels["sessions"]
channels = channels.sort_values("revenue", ascending=False).reset_index(drop=True)
channels

In [ ]:
# The AI's platforms cell, fixed: one row per date, platform and campaign before the merge
revenue = sessions.groupby(["date", "session_campaign"], as_index=False)[["sessions", "transactions", "revenue"]].sum()
revenue = revenue.rename(columns={"session_campaign": "campaign"})
revenue["campaign"] = revenue["campaign"].str.strip().str.lower()

daily_spend = spend.groupby(["date", "platform", "campaign"], as_index=False)["spend"].sum()
campaign_days = daily_spend.merge(revenue, on=["date", "campaign"], how="left", validate="one_to_one")
platforms = campaign_days.groupby("platform")[["spend", "revenue"]].sum()
platforms["roas"] = platforms["revenue"] / platforms["spend"]
platforms.round(2)

In [ ]:
# Your checks: topic 14.3.'s function, and the assert
def validate_report(sessions, report, start, end):
    """Return each check's name and whether the report passes it."""
    return {
        "every day is there": sessions["date"].nunique() == len(pd.date_range(start, end)),
        "sessions add up": bool(report["sessions"].sum() == sessions["sessions"].sum()),
        "revenue adds up": bool(round(report["revenue"].sum(), 2) == round(sessions["revenue"].sum(), 2)),
        "rates from totals": bool(((report["conversion_rate"] - report["transactions"] / report["sessions"]).abs() < 0.000001).all()),
    }


checks = validate_report(sessions, channels, START, END)
assert all(checks.values()), checks
checks

**Summary.** In Q1 2025, Google Ads returned \$2.47 for every dollar it spent, Meta \$2.33 and YouTube \$1.89. Meta doesn't beat Google Ads, so the case for moving budget to it is gone. YouTube returns the least, and that's where I'd look first. Among channel groups, Email converts best, at 6.22% of sessions.

In [ ]:
# @title Check your work
# @markdown This cell runs last, with the others. Run it with Runtime → Restart session and run all.
# This cell only reads your variables and the cells that have run. It doesn't change anything.
def check_your_work():
    import sys
    import pandas as pd
    found = globals()
    results = []

    inputs = found.get("In")
    history = [source for source in inputs[1:-1] if "def check_your_work" not in source] if isinstance(inputs, list) else None
    checked_before = isinstance(inputs, list) and any("def check_your_work" in source for source in inputs[1:-1])
    error = getattr(sys, "last_exc", None) or getattr(sys, "last_value", None)
    message = f"{type(error).__name__}: {error}" if error is not None else ""
    if history is None:
        results.append((None, "This check can't see which cells have run, so it can't tell whether this was a clean run. Choose Runtime → Restart session and run all to be sure."))
    elif checked_before:
        results.append((None, "This cell has run before in this session, so the cells above may have changed since they ran. Choose Runtime → Restart session and run all to check the notebook as it is now."))
    elif [source for source in history if source.strip() and history.count(source) > 1]:
        repeated = [source for source in history if source.strip() and history.count(source) > 1]
        first_line = repeated[0].strip().splitlines()[0]
        results.append((None, f"Some cells ran more than once in this session, such as the one that starts with {first_line!r}. Choose Runtime → Restart session and run all, so every cell runs once, from the top."))
    elif "AssertionError" in message:
        results.append((None, f"Your assert stopped the notebook: {error}. A check failed, so a table is still wrong: the ❌ below say which. Fix it, then choose Runtime → Restart session and run all."))
    elif "one-to-one" in message:
        results.append((None, "The merge stopped: a date and campaign appear more than once. That's what validate=\"one_to_one\" is for. Total the spend by date, platform and campaign before you merge (step 5)."))
    elif error is not None:
        results.append((None, f"A cell stopped with {type(error).__name__} earlier in this session, so the cells after it may not have run in order. Choose Runtime → Restart session and run all."))
    else:
        results.append((True, "Every cell in this session ran once, with no errors."))

    sessions = found.get("sessions")
    if not isinstance(sessions, pd.DataFrame) or [column for column in ["date", "sessions", "revenue", "channel_group"] if column not in sessions.columns]:
        results.append((False, "sessions should be the clean sessions table, with a channel_group column. Fix the AI's first cell where it is, and keep its last lines."))
    else:
        days, total = sessions["date"].nunique(), int(sessions["sessions"].sum())
        if days == 81 or total == 305686:
            results.append((False, f"The sessions table has {days} of Q1's 90 days and {total:,} sessions: March 2025 is short. Its export writes dates day first, such as 05/03/2025, and format=\"mixed\" reads that as May 3. Read March 2025 with format=\"%d/%m/%Y\" (step 3)."))
        elif days != 90 or total != 324563 or abs(float(sessions["revenue"].sum()) - 655182.61) > 0.05:
            results.append((False, f"Q1 2025 has 90 days, 324,563 sessions and $655,182.61 of revenue. The sessions table has {days} days, {total:,} sessions and ${float(sessions['revenue'].sum()):,.2f}. Check how each month's dates are read (step 3)."))
        else:
            results.append((True, "The sessions table has every day of Q1 2025: 324,563 sessions and $655,182.61 of revenue."))

    channels = found.get("channels")
    columns = ["channel_group", "sessions", "transactions", "revenue", "conversion_rate"]
    if not isinstance(channels, pd.DataFrame) or [column for column in columns if column not in channels.columns]:
        results.append((False, f"channels should be a DataFrame with the columns {', '.join(columns)}."))
    elif len(channels) != 9 or int(channels["sessions"].sum()) != 324563:
        results.append((False, f"The channels table should have 9 channel groups and 324,563 sessions. It has {len(channels)} and {int(channels['sessions'].sum()):,}. Fix the sessions table first (step 3)."))
    else:
        email = channels.loc[channels["channel_group"] == "Email", "conversion_rate"]
        pooled = channels["transactions"] / channels["sessions"]
        if ((channels["conversion_rate"] - pooled).abs() > 0.00001).any():
            rate = float(email.iloc[0]) if len(email) else float("nan")
            results.append((False, f"The conversion rates aren't transactions divided by sessions: Email's is {rate:.2%}, and its totals give {1659 / 26659:.2%}. That's a mean of daily rates. Divide the totals after grouping (step 4)."))
        else:
            results.append((True, "9 channel groups, with conversion rates from totals: Email converts best, at 6.22%."))

    platforms = found.get("platforms")
    expected = {"Google Ads": 2.4695, "Meta": 2.3315, "YouTube": 1.8937}
    if not isinstance(platforms, pd.DataFrame) or "roas" not in platforms.columns:
        results.append((False, "platforms should be a DataFrame with a roas column, and one row per platform."))
    else:
        table = platforms.set_index("platform") if "platform" in platforms.columns else platforms
        roas = {name: float(table.loc[name, "roas"]) for name in expected if name in table.index}
        merged_with_check = isinstance(inputs, list) and any("one_to_one" in source for source in inputs[1:-1])
        if len(roas) != 3:
            results.append((False, "The platforms table should have one row per platform: Google Ads, Meta and YouTube."))
        elif roas["Meta"] > 4:
            results.append((False, f"Meta's ROAS is {roas['Meta']:.2f}. Meta's export has a row per ad set, so the merge copied each campaign-day's revenue to every ad set. Total the spend by date, platform and campaign before you merge (step 5)."))
        elif [name for name in expected if abs(roas[name] - expected[name]) > 0.0005]:
            results.append((False, "The ROAS should be Google Ads 2.47, Meta 2.33 and YouTube 1.89. Fix the sessions table first: the revenue comes from it (step 3)."))
        elif not merged_with_check:
            results.append((False, "The ROAS is right. Make the merge stop if a date and campaign appear more than once: validate=\"one_to_one\" (step 5)."))
        else:
            results.append((True, "ROAS by platform: Google Ads 2.47, Meta 2.33 and YouTube 1.89. Meta doesn't beat Google Ads."))

    validate_report, checks = found.get("validate_report"), found.get("checks")
    asserted = isinstance(inputs, list) and any(line.strip().startswith("assert ") and "checks" in line for source in inputs[1:-1] for line in source.splitlines())
    if not callable(validate_report):
        results.append((False, "Paste topic 14.3.'s validate_report(sessions, report, start, end) into the Your checks section (step 2)."))
    elif not isinstance(checks, dict) or not checks:
        results.append((False, "Create checks: validate_report(sessions, channels, START, END) (step 6)."))
    elif not all(checks.values()):
        failed = [name for name, passed in checks.items() if not passed]
        results.append((False, f"These checks still fail: {', '.join(failed)}. Fix the tables they point to."))
    elif isinstance(sessions, pd.DataFrame) and isinstance(channels, pd.DataFrame) and all(validate_report(sessions[~(sessions["date"].dt.month.eq(3) & sessions["date"].dt.day.between(4, 12))], channels, "2025-01-01", "2025-03-31").values()):
        results.append((False, "validate_report() passes a sessions table that's missing March 4 to 12. Use topic 14.3.'s function as it is."))
    elif not asserted:
        results.append((False, "Every check passes. Make a failure stop the notebook: assert all(checks.values()), checks (step 6)."))
    else:
        results.append((True, "Every check in checks passes, and the assert stops the notebook if one doesn't."))

    for passed, message in results:
        print({True: "✅ ", False: "❌ ", None: "⚠️ "}[passed] + message)
    if any(passed is False for passed, message in results):
        print("\nFix the first ❌, then choose Runtime → Restart session and run all.")
    elif any(passed is None for passed, message in results):
        print("\nYour results match. Restart the session and run all, to check them in a clean run.")
    else:
        print("\nEverything matches, in a clean run.")


check_your_work()

## Hints

Stuck on a step? Open its hint. Each one says where the course showed it, and gives the shape of the code, with `...` where your part goes. Try the step from the hint before you open the solution.

<details>
<summary><b>Step 2: the checks</b></summary>

Topic 14.3.: copy the function from your 14.3. workbook as it is. Run it without `assert` first, so you can see every check's answer.

~~~python
def validate_report(sessions, report, start, end):
    """Return each check's name and whether the report passes it."""
    return {
        "every day is there": sessions["date"].nunique() == len(pd.date_range(start, end)),
        "sessions add up": ...,
        "revenue adds up": ...,
        "rates from totals": ...,
    }


validate_report(sessions, channels, START, END)
~~~

</details>

<details>
<summary><b>Step 3: the missing days</b></summary>

Count the different dates in each month. Q1 2025 has 31, 28 and 31 days.

~~~python
sessions.groupby(sessions["date"].dt.month)["date"].nunique()
~~~

One month is short. Read its file as it comes, without the AI's cell, and look at the `date` column:

~~~python
pd.read_csv(DATA_URL + "sessions_export_2025_...csv").head()
~~~

`05/03/2025` is March 5, written day first. The AI's `format="mixed"` guesses each date on its own and reads it month first, as May 3. That's why the days go missing without an error. Topic 3.2. read this file with its own format. In the AI's first cell, replace the `pd.to_datetime()` line:

~~~python
DAY_FIRST_MONTHS = ["2025-..."]  # put this line above the loop
...
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
~~~

</details>

<details>
<summary><b>Step 4: rates from totals</b></summary>

Topic 4.4.: a channel group's conversion rate is its transactions divided by its sessions, after grouping. A mean of daily rates gives a quiet day as much say as a busy one. Replace the AI's channels cell with a plain `groupby()`:

~~~python
channels = sessions.groupby("channel_group", as_index=False)[["sessions", "transactions", "revenue"]].sum()
channels["conversion_rate"] = ... / ...
channels = channels.sort_values("revenue", ascending=False).reset_index(drop=True)
~~~

</details>

<details>
<summary><b>Step 5: the duplicated revenue</b></summary>

Topic 14.2.'s first question, for each platform: the rows, and the different date and campaign pairs.

~~~python
spend.groupby("platform").size(), spend.drop_duplicates(["date", "platform", "campaign"]).groupby("platform").size()
~~~

Meta has about 2.6 rows for each pair, so the merge copies each campaign-day's revenue to every ad set. Topic 5.4.: in the AI's platforms cell, total the spend first, and merge that.

~~~python
daily_spend = spend.groupby([..., ..., ...], as_index=False)["spend"].sum()
campaign_days = daily_spend.merge(revenue, on=["date", "campaign"], how="left", validate=...)
~~~

</details>

<details>
<summary><b>Step 6: stop the notebook</b></summary>

Topic 14.3.: with the three fixes in place, every check passes, and the `assert` does nothing. If a check fails on a later run, the notebook stops here.

~~~python
checks = validate_report(sessions, channels, START, END)
assert all(checks.values()), checks
~~~

</details>

<details>
<summary><b>Show solution</b></summary>

Try the exercise yourself first. This is one way to solve it: if the check passes, your way is right too.

```python
# The AI's first cell, fixed: March 2025's export writes its dates day first
DAY_FIRST_MONTHS = ["2025-03"]
frames = []
for month in MONTHS:
    export = pd.read_csv(DATA_URL + f"sessions_export_{month.replace('-', '_')}.csv",
                         na_values={"session_campaign": ["(not set)", "-"]})
    date_format = "%d/%m/%Y" if month in DAY_FIRST_MONTHS else "%Y-%m-%d"
    export["date"] = pd.to_datetime(export["date"], format=date_format)
    # Some exports store revenue as currency text, such as "$1,234.50"
    if not pd.api.types.is_numeric_dtype(export["revenue"]):
        export["revenue"] = export["revenue"].str.replace("$", "").str.replace(",", "").astype(float)
    frames.append(export)

raw = pd.concat(frames, ignore_index=True)
raw["session_campaign"] = raw["session_campaign"].fillna("(not set)")
deduped = raw.drop_duplicates()  # The monthly exports overlap by two days
sessions = deduped[deduped["date"].between(START, END) & (deduped["session_medium"] != "test")].reset_index(drop=True)
sessions["channel_group"] = sessions.apply(row_channel_group, axis=1)  # The team's channel groups
sessions.shape
```

```python
# The AI's channels cell, fixed: rates from totals
channels = sessions.groupby("channel_group", as_index=False)[["sessions", "transactions", "revenue"]].sum()
channels["conversion_rate"] = channels["transactions"] / channels["sessions"]
channels = channels.sort_values("revenue", ascending=False).reset_index(drop=True)
channels
```

```python
# The AI's platforms cell, fixed: one row per date, platform and campaign before the merge
revenue = sessions.groupby(["date", "session_campaign"], as_index=False)[["sessions", "transactions", "revenue"]].sum()
revenue = revenue.rename(columns={"session_campaign": "campaign"})
revenue["campaign"] = revenue["campaign"].str.strip().str.lower()

daily_spend = spend.groupby(["date", "platform", "campaign"], as_index=False)["spend"].sum()
campaign_days = daily_spend.merge(revenue, on=["date", "campaign"], how="left", validate="one_to_one")
platforms = campaign_days.groupby("platform")[["spend", "revenue"]].sum()
platforms["roas"] = platforms["revenue"] / platforms["spend"]
platforms.round(2)
```

```python
# Your checks: topic 14.3.'s function, and the assert
def validate_report(sessions, report, start, end):
    """Return each check's name and whether the report passes it."""
    return {
        "every day is there": sessions["date"].nunique() == len(pd.date_range(start, end)),
        "sessions add up": bool(report["sessions"].sum() == sessions["sessions"].sum()),
        "revenue adds up": bool(round(report["revenue"].sum(), 2) == round(sessions["revenue"].sum(), 2)),
        "rates from totals": bool(((report["conversion_rate"] - report["transactions"] / report["sessions"]).abs() < 0.000001).all()),
    }


checks = validate_report(sessions, channels, START, END)
assert all(checks.values()), checks
checks
```

```markdown
**Summary.** In Q1 2025, Google Ads returned \$2.47 for every dollar it spent, Meta \$2.33 and YouTube \$1.89. Meta doesn't beat Google Ads, so the case for moving budget to it is gone. YouTube returns the least, and that's where I'd look first. Among channel groups, Email converts best, at 6.22% of sessions.
```

</details>